# Exercise 03 — batch producer & partition keys

**Goal:** send many events at once and **see, in your own output**, the consequence of choosing a key (or not).

Why care? In real systems, ordering matters. If you process payments, the *withdraw* event must arrive before the *deposit* — otherwise the balance temporarily goes negative. Kafka guarantees order **per partition**, so you must put related events on the same partition by giving them the same key.

In [ ]:
from confluent_kafka import Producer
from collections import defaultdict, Counter
import json, time, random

producer = Producer({
    'bootstrap.servers': 'redpanda:29092',
    'client.id':         'batch-producer',
})

houses = ['haus_a', 'haus_b', 'haus_c']
topics = ['strom', 'wasser']

## Step 1 — send 15 random events

Each event picks a random house and topic. We capture the broker's *delivery report* (which contains the assigned partition) so we can analyse the mapping after the fact.

**Watch:** the same `key` always returns the same `partition`, regardless of how often you re-run.

In [ ]:
results = []

def delivery_report(err, msg):
    if err:
        print(f'FAILED: {err}')
    else:
        results.append({
            'topic':     msg.topic(),
            'partition': msg.partition(),
            'key':       msg.key().decode(),
        })
        print(f'  -> {msg.topic()} [P{msg.partition()}] '
              f'key={msg.key().decode()} offset={msg.offset()}')

for _ in range(15):
    house = random.choice(houses)
    topic = random.choice(topics)
    value = round(random.uniform(1.0, 100.0), 2)
    event = json.dumps({
        'sensor':    topic,
        'haus':      house,
        'wert':      value,
        'einheit':   'kWh' if topic == 'strom' else 'Liter',
        'timestamp': time.time(),
    })
    producer.produce(topic, key=house.encode(), value=event.encode(),
                     callback=delivery_report)

producer.flush()
print('15 events sent.')

## Step 2 — analyse the key→partition mapping

We group the deliveries by `(topic, key)` and check whether each pair always landed on the same partition. If `consistent? = yes`, the key-hashing is doing its job.

In [ ]:
key_partitions = defaultdict(set)
for r in results:
    key_partitions[(r['topic'], r['key'])].add(r['partition'])

print(f'{"Topic":<7} | {"Key":<7} | Partition(s) | Consistent?')
print('-' * 50)
for (t, k), parts in sorted(key_partitions.items()):
    plist = ', '.join(str(p) for p in sorted(parts))
    flag  = 'yes' if len(parts) == 1 else 'NO!'
    print(f'{t:<7} | {k:<7} | {plist:<12} | {flag}')

## Task A — events without a key

**Send 6 events to `strom` with `key=None`.** Where do they land?

**Expected:** the producer falls back to a *sticky* round-robin — it fills one partition for a while, then switches. So you'll typically see all 6 on the same partition, *but* there's no guarantee — if you re-run, the partition can change.

**Consequence:** keyless events have **no ordering guarantee** between each other. If order matters, give them a key.

In [ ]:
no_key_partitions = []

def no_key_cb(err, msg):
    if not err:
        no_key_partitions.append(msg.partition())
        print(f'  -> P{msg.partition()} offset={msg.offset()}')

# TODO: loop 6 times, produce a small JSON event with key=None

producer.flush()
print(f'Distribution: {dict(Counter(no_key_partitions))}')

## Task B — confirm in the Console

1. Open Redpanda Console → topic `strom` → *Messages*.
2. Use the *Partition* filter — first **0** only, then **1** only.
3. Compare the keys you see with what your Python output reported    above. They should match exactly.

## What you learned

- Choosing a key gives **deterministic, ordered** routing inside a   topic.
- No key → load is spread but **order is lost** between events.
- The same key always lands on the same partition, **as long as the   partition count doesn't change**. Re-creating a topic with a   different partition count will reshuffle everything.